# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gamal1osama/flyrank/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.


## 1. Unit of analysis + time window

### The Contract in Plain Words (Five Core Answers)

1. **What one row means for your lane:**
   - In the raw warehouse fact table (`fact_content_daily_performance`): **One row = one pseudonymized content item (`content_hash_id`) for a specific client (`client_hash_id`) on a single calendar day (`report_date`).**
   - In our lane's aggregated decision frame: **One row = one content item per client evaluated at decision cutoff timestamp $t = \text{2026-03-20}$ over a trailing feature window.**

2. **Which table(s) you'll use:**
   - Primary Fact Table: `fact_content_daily_performance` (partitioned by month, evaluated on mid-panel month `month=2026-03`).
   - Dimension Tables: `dim_clients` (for tracking start dates and access profiles) and `dim_content` (for content metadata and joins).

3. **Which time window:**
   - **Evaluation Partition:** Mid-panel month `month=2026-03` (**March 1, 2026 to March 31, 2026**).
   - **Feature Observation Window:** **2026-03-01 to 2026-03-20** (20 days prior to decision cutoff).
   - **Target Outcome Window:** **2026-03-21 to 2026-03-31** (forward 11 days strictly after decision cutoff).

4. **What you'd predict or rank (label or proxy):**
   - **Target:** Forward-window traffic decay — whether an established, search-visible page experiences a $\ge 20\%$ decline in search impressions in the forward window relative to baseline demand (`imp_target < 0.80 * imp_mid`) among active pages (`imp_mid >= 100`).
   - **Output:** Continuous priority score ranking candidate pages into a weekly editorial review queue.

5. **One thing you deliberately exclude:**
   - **Deliberately Excluded:** Any metric computed from dates after the decision cutoff ($t > \text{2026-03-20}$) such as `imp_target` or forward percentage changes, as well as pseudonymized identifiers (`client_hash_id`, `content_hash_id`) and product decision flags.


In [1]:
import os, getpass
from pathlib import Path
import duckdb
import pandas as pd
import numpy as np

# Safe Hugging Face token resolution (Colab Secret -> env var -> ~/.cache/huggingface/token -> prompt)
# Token is never hardcoded or printed in public cells
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
if not HF_TOKEN:
    cached_token = Path.home() / ".cache/huggingface/token"
    if cached_token.exists():
        HF_TOKEN = cached_token.read_text().strip()
if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Enter your Hugging Face READ token: ")

# Initialize DuckDB connection to Hugging Face
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
try:
    con.execute("SET enable_progress_bar = false")
except Exception:
    pass

REL = "hf://datasets/FlyRank/internship-warehouse"
SRC_DAILY_MARCH = f"{REL}/fact_content_daily_performance/month=2026-03/*.parquet"

print("DuckDB connected successfully to Hugging Face warehouse.")
print(f"Data Source: {SRC_DAILY_MARCH}")



DuckDB connected successfully to Hugging Face warehouse.
Data Source: hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet


## 2. Fields: feature / label / context / excluded

Every field touched by our pipeline belongs to exactly one of four strict contract buckets:

| Field Name | Contract Bucket | Window / Definition | One-Line Rationale |
|---|---|---|---|
| `imp_early` | **Feature** | `report_date` 2026-03-01 to 2026-03-10 | Trailing search impressions; fully known 10 days before decision cutoff. |
| `imp_mid` | **Feature** | `report_date` 2026-03-11 to 2026-03-20 | Recent baseline search impressions; fully accrued up to decision cutoff $t$. |
| `clk_mid` | **Feature** | `report_date` 2026-03-11 to 2026-03-20 | Search clicks in recent baseline; observed search demand before cutoff $t$. |
| `pos_mid` | **Feature** | `report_date` 2026-03-11 to 2026-03-20 | Average GSC ranking position during baseline window; known at cutoff $t$. |
| `active_days_mid` | **Feature** | `report_date` 2026-03-11 to 2026-03-20 | Count of active impression days; measures search consistency before cutoff $t$. |
| `is_declining_target` | **Label / Proxy** | `report_date` 2026-03-21 to 2026-03-31 | Target: forward impression decline $\ge 20\%$. Computed strictly in the forward window; never a feature. |
| `client_hash_id` | **Context** | Dimension / join key | Scrambled client identifier used strictly for grouped cross-validation splits. |
| `content_hash_id` | **Context** | Dimension / join key | Scrambled content identifier used for joins and review queue tracking; never a feature. |
| `leaky_forward_pct` | **Excluded** | `report_date` > 2026-03-20 | **Target Leakage:** Directly encodes forward outcome; strictly banned from honest models. |
| `ga4_sessions` (unfiltered) | **Excluded** | Full month | **Unbalanced Sparsity:** 95.8% of daily rows lack GA4; zeros represent tracking absence, not zero traffic. |


In [2]:
# Summary view of Field Classification Contract
classification_summary = pd.DataFrame([
    {"Bucket": "Feature", "Count": 5, "Fields": "imp_early, imp_mid, clk_mid, pos_mid, active_days_mid", "Usage Rule": "Knowable strictly prior to decision timestamp t"},
    {"Bucket": "Label / Proxy", "Count": 1, "Fields": "is_declining_target", "Usage Rule": "Computed strictly in forward window t+1 to t+30; NEVER a feature"},
    {"Bucket": "Context", "Count": 2, "Fields": "client_hash_id, content_hash_id", "Usage Rule": "Grouping and joining only; never fed directly to model"},
    {"Bucket": "Excluded", "Count": 2, "Fields": "leaky_forward_pct, unfiltered ga4 metrics", "Usage Rule": "Banned due to future target leakage or missing tracking"},
])
print("=== Search Intelligence Field Classification Contract ===")
print(classification_summary.to_string(index=False))



=== Search Intelligence Field Classification Contract ===
       Bucket  Count                                                Fields                                                       Usage Rule
      Feature      5 imp_early, imp_mid, clk_mid, pos_mid, active_days_mid                  Knowable strictly prior to decision timestamp t
Label / Proxy      1                                   is_declining_target Computed strictly in forward window t+1 to t+30; NEVER a feature
      Context      2                       client_hash_id, content_hash_id           Grouping and joining only; never fed directly to model
     Excluded      2             leaky_forward_pct, unfiltered ga4 metrics          Banned due to future target leakage or missing tracking


## 3. Verify it with queries (grain, counts, missing values, windows)

Here we prove every claim in our contract with real SQL queries against `month=2026-03` of the warehouse fact table:

1. **Query 1 (The Grain):** Prove that `(report_date, client_hash_id, content_hash_id)` is 100% unique (a `HAVING count(*) > 1` query returns zero rows).
2. **Query 2 (Counts & Date Span):** Prove the exact row count and date boundaries of the mid-panel slice.
3. **Query 3 (Availability with IS TRUE):** Prove availability by filtering with `ga4_data_available IS TRUE` and `gsc_data_available IS TRUE`.
4. **Five Features Frame:** Construct the 5-feature dataset where every feature has a documented "available when?" justification.
5. **The Trap (Deliberate Leakage Experiment):** Inject one label-derived column (`leaky_forward_pct`), show Precision@50 artificially leaping to 1.000 (100%), inspect the cheat tree, then delete the leaky feature to retain the honest baseline.


In [3]:
# === 1. Three Verification Queries ===

# Query 1: The Grain (one row really is report_date x client x content)
q1 = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) as c
    FROM read_parquet('{SRC_DAILY_MARCH}')
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING c > 1
    LIMIT 5
""").df()
print(f"Query 1 (Grain Check): {len(q1)} duplicate key combinations found. (Expected: 0 -> Grain holds strictly!)")

# Query 2: Slice Row Count and Date Span
q2 = con.sql(f"""
    SELECT 
        COUNT(*) as total_rows, 
        MIN(report_date) as min_date, 
        MAX(report_date) as max_date,
        COUNT(DISTINCT client_hash_id) as active_clients,
        COUNT(DISTINCT content_hash_id) as active_content_items
    FROM read_parquet('{SRC_DAILY_MARCH}')
""").df()
print(f"\nQuery 2 (Slice Dimensions & Date Span):")
print(q2.to_string(index=False))

# Query 3: Availability (Filtered with IS TRUE)
q3 = con.sql(f"""
    SELECT 
        COUNT(*) as total_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) as ga4_available_rows,
        ROUND(COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) * 100.0 / COUNT(*), 2) as ga4_available_pct,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) as gsc_available_rows,
        ROUND(COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) * 100.0 / COUNT(*), 2) as gsc_available_pct
    FROM read_parquet('{SRC_DAILY_MARCH}')
""").df()
print(f"\nQuery 3 (Availability with IS TRUE):")
print(q3.to_string(index=False))

# === 2. Build the 5-Feature Frame and Forward Target ===
feature_query = f"""
WITH windowed AS (
    SELECT 
        client_hash_id,
        content_hash_id,
        -- Feature 1: Early window impressions (March 1-10)
        SUM(CASE WHEN report_date <= '2026-03-10' THEN gsc_impressions ELSE 0 END) AS imp_early,
        -- Feature 2: Mid window impressions (March 11-20)
        SUM(CASE WHEN report_date > '2026-03-10' AND report_date <= '2026-03-20' THEN gsc_impressions ELSE 0 END) AS imp_mid,
        -- Feature 3: Mid window clicks (March 11-20)
        SUM(CASE WHEN report_date > '2026-03-10' AND report_date <= '2026-03-20' THEN gsc_clicks ELSE 0 END) AS clk_mid,
        -- Feature 4: Mid window average position (March 11-20)
        AVG(CASE WHEN report_date > '2026-03-10' AND report_date <= '2026-03-20' AND gsc_avg_position > 0 THEN gsc_avg_position ELSE NULL END) AS pos_mid,
        -- Feature 5: Mid window active search days (March 11-20)
        COUNT(DISTINCT CASE WHEN report_date > '2026-03-10' AND report_date <= '2026-03-20' AND gsc_impressions > 0 THEN report_date ELSE NULL END) AS active_days_mid,
        
        -- Target Window (March 21-31)
        SUM(CASE WHEN report_date > '2026-03-20' THEN gsc_impressions ELSE 0 END) AS imp_target
    FROM read_parquet('{SRC_DAILY_MARCH}')
    GROUP BY client_hash_id, content_hash_id
    HAVING imp_mid >= 100
)
SELECT 
    client_hash_id,
    content_hash_id,
    imp_early,
    imp_mid,
    clk_mid,
    COALESCE(pos_mid, 0.0) as pos_mid,
    active_days_mid,
    -- Honest Forward Label: Did impressions drop by >= 20% in forward window?
    CASE WHEN imp_target < 0.80 * imp_mid THEN 1 ELSE 0 END AS is_declining_target,
    -- THE LEAK TRAP: Forward percentage change computed from future outcome!
    ROUND((imp_target - imp_mid) * 100.0 / (imp_mid + 1), 2) AS leaky_forward_pct
FROM windowed
"""

df_frame = con.sql(feature_query).df()
print(f"\nExtracted Feature Frame: {len(df_frame):,} content items with baseline demand (imp_mid >= 100).")
print(f"Base Decline Rate: {df_frame['is_declining_target'].mean():.1%}")

# Display feature documentation
print("\n--- Five Features: 'Knowable at the decision moment because...' ---")
print("1. imp_early: Knowable because it was measured in GSC during March 1-10, fully 10 days before decision cutoff.")
print("2. imp_mid: Knowable because it accrued up to March 20, the exact decision timestamp.")
print("3. clk_mid: Knowable because it reflects user clicks accrued during the baseline window before cutoff.")
print("4. pos_mid: Knowable because it represents average SERP ranking position recorded by GSC before cutoff.")
print("5. active_days_mid: Knowable because it counts days with active impressions during the baseline window.")

# === 3. The Trap: Deliberate Leakage Experiment ===
from sklearn.tree import DecisionTreeClassifier, export_text

def precision_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

y = df_frame["is_declining_target"].values
honest_features = ["imp_early", "imp_mid", "clk_mid", "pos_mid", "active_days_mid"]
X_honest = df_frame[honest_features]

# Train honest tree
tree_honest = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42)
tree_honest.fit(X_honest, y)
honest_p50 = precision_at_k(tree_honest.predict_proba(X_honest)[:, 1], y, 50)

# Train leaky tree (with leaky_forward_pct)
X_leaky = df_frame[honest_features + ["leaky_forward_pct"]]
tree_leaky = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42)
tree_leaky.fit(X_leaky, y)
leaky_p50 = precision_at_k(tree_leaky.predict_proba(X_leaky)[:, 1], y, 50)

print("\n=== The Deliberate Leakage Experiment ===")
print(f"Leaky Model Precision@50:  {leaky_p50:.3f}  <- ARTIFICIAL PERFECTION (100% cheated!)")
print(f"Honest Model Precision@50: {honest_p50:.3f}  <- REALISTIC BASELINE (without peeking)")
print("\nInspection of Leaky Decision Tree Rules:")
print(export_text(tree_leaky, feature_names=honest_features + ["leaky_forward_pct"]))
print("Verdict: The leaky tree bypassed all search signals and split solely on the target formula (<= -19.99%).")
print("Action Taken: Deleted 'leaky_forward_pct' permanently from feature pipeline.")



Query 1 (Grain Check): 0 duplicate key combinations found. (Expected: 0 -> Grain holds strictly!)



Query 2 (Slice Dimensions & Date Span):
 total_rows   min_date   max_date  active_clients  active_content_items
    9841378 2026-03-01 2026-03-31              55                331437



Query 3 (Availability with IS TRUE):
 total_rows  ga4_available_rows  ga4_available_pct  gsc_available_rows  gsc_available_pct
    9841378              413966               4.21             3611061              36.69



Extracted Feature Frame: 71,569 content items with baseline demand (imp_mid >= 100).
Base Decline Rate: 19.8%

--- Five Features: 'Knowable at the decision moment because...' ---
1. imp_early: Knowable because it was measured in GSC during March 1-10, fully 10 days before decision cutoff.
2. imp_mid: Knowable because it accrued up to March 20, the exact decision timestamp.
3. clk_mid: Knowable because it reflects user clicks accrued during the baseline window before cutoff.
4. pos_mid: Knowable because it represents average SERP ranking position recorded by GSC before cutoff.
5. active_days_mid: Knowable because it counts days with active impressions during the baseline window.



=== The Deliberate Leakage Experiment ===
Leaky Model Precision@50:  1.000  <- ARTIFICIAL PERFECTION (100% cheated!)
Honest Model Precision@50: 0.440  <- REALISTIC BASELINE (without peeking)

Inspection of Leaky Decision Tree Rules:
|--- leaky_forward_pct <= -19.99
|   |--- class: 1
|--- leaky_forward_pct >  -19.99
|   |--- class: 0

Verdict: The leaky tree bypassed all search signals and split solely on the target formula (<= -19.99%).
Action Taken: Deleted 'leaky_forward_pct' permanently from feature pipeline.


## 4. Data limits

What can this warehouse data never tell you? A rigorous data contract acknowledges its empirical boundaries:

1. **Unbalanced Panel Coverage:**
   - Client tracking histories do not share a common starting date (`gsc_data_start` and `ga4_data_start` differ by client). Only a subset of clients have 12+ months of continuous history; cross-client seasonality cannot be uniformly assumed.

2. **Severe GA4 Analytics Sparsity (4.21% Availability):**
   - In `month=2026-03`, only **413,966 out of 9,841,378 daily rows (4.21%)** have `ga4_data_available IS TRUE`.
   - Rows prior to a client's GA4 setup have zeroes filled in with `ga4_data_available = FALSE`. Treating these zeroes as "zero user engagement" is a major data error. GA4 metrics must always be gated with `WHERE ga4_data_available IS TRUE`.

3. **Observational Nature — No Causal Recovery Proof:**
   - The data records observational logs from search engines and analytics. It cannot prove that refreshing a page *caused* rank preservation or recovery; such claims require randomized A/B experimentation.

4. **Window Alignment & Ingestion Lags:**
   - Production Google Search Console data carries a 2–3 day ingestion delay. Any operational pipeline must maintain a freshness buffer to avoid evaluating incomplete recent days.


In [4]:
# Prove the Unbalanced Panel and GA4 Sparsity with a Verification Query
dim_clients_src = f"{REL}/dim_clients.parquet"
client_limits = con.sql(f"""
    SELECT 
        COUNT(*) as total_clients,
        COUNT(*) FILTER (WHERE gsc_data_start IS NOT NULL) as clients_with_gsc,
        COUNT(*) FILTER (WHERE ga4_data_start IS NOT NULL) as clients_with_ga4,
        COUNT(*) FILTER (WHERE gsc_data_start <= '2025-06-30') as clients_with_1yr_gsc,
        MIN(gsc_data_start) as earliest_gsc_start,
        MAX(gsc_data_start) as latest_gsc_start
    FROM read_parquet('{dim_clients_src}')
""").df()

print("=== Warehouse Panel Limitations (from dim_clients) ===")
print(client_limits.to_string(index=False))
print("\nProof: History start dates vary widely across clients, confirming an unbalanced panel.")
print("Recommendation: Always verify per-client tracking boundaries before aggregating across time.")



=== Warehouse Panel Limitations (from dim_clients) ===
 total_clients  clients_with_gsc  clients_with_ga4  clients_with_1yr_gsc earliest_gsc_start latest_gsc_start
           104                67                51                     9         2025-01-27       2026-06-02

Proof: History start dates vary widely across clients, confirming an unbalanced panel.
Recommendation: Always verify per-client tracking boundaries before aggregating across time.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
